# 01. L4 の確認と、小さいOSSモデルを1回動かす

このノートは、[colab-oss-lab](https://github.com/moruku36/colab-oss-lab) の最初の実験です。

やることは2つだけです。

1. 今つながっているGPUが L4 かどうか見る
2. 小さい公開モデル（Qwen2.5-1.5B-Instruct）に、日本語で1問だけ聞く

所要時間の目安: 初回ダウンロード込みで 5〜15 分。
このモデルは約1.5B（15億パラメータ）です。L4 なら圧縮しなくても載ります。

---

## 実行する前に

1. 上のメニュー **ランタイム → ランタイムのタイプを変更**
2. Hardware accelerator を **L4 GPU** にする
3. Save
4. このセルより下を、上から順に ▶ で実行する

ターミナル（黒い画面）には貼らないでください。**コードセル**で実行します。


## 1. GPU を確認する

In [ ]:
import subprocess
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise SystemExit(
        "GPUにつながっていません。"
        "ランタイム → ランタイムのタイプを変更 → L4 GPU → Save のあと、"
        "このセルをもう一度実行してください。"
    )

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print("GPU name:", gpu_name)
print("VRAM GB :", round(vram_gb, 1))

smi = subprocess.check_output(["nvidia-smi"], text=True)
print()
print("----- nvidia-smi -----")
print(smi)

gpu_ok = "L4" in gpu_name
if gpu_ok:
    print("L4判定: はい。普段使いの机です。")
else:
    print("L4判定: いいえ。名前は", gpu_name)


## 2. 小さいモデルを読み込む

使うモデル: `Qwen/Qwen2.5-1.5B-Instruct`

- 公開されている会話用モデル
- 日本語も一応出る
- サイズが小さいので、最初の確認に向く
- 「一番賢いモデル」ではない。動くことを確かめるための練習用

初回は Hugging Face からファイルを取ります。少し待ちます。


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype="auto",
    device_map="auto",
)

print("読み込み完了:", MODEL_ID)
print("device:", next(model.parameters()).device)
print("dtype :", next(model.parameters()).dtype)


## 3. 日本語で1問だけ聞く

返事が短くても、多少おかしくても構いません。
このサイズでは、Gemini より雑なのが普通です。


In [ ]:
prompt = "小学校の児童にも分かる言葉で、GPUとVRAMの違いを3文で説明してください。"

messages = [
    {"role": "system", "content": "You are a helpful assistant. Answer in Japanese."},
    {"role": "user", "content": prompt},
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)
inputs = tokenizer([text], return_tensors="pt").to(model.device)

out = model.generate(
    **inputs,
    max_new_tokens=160,
    do_sample=False,
)
reply = tokenizer.decode(
    out[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True,
)

print("質問:")
print(prompt)
print()
print("返事:")
print(reply.strip())


## 4. 結果をこのノートの下に出す

次のセルは、GitHub の `docs/results/01_first_run.md` に貼れる文章を作ります。
出力をコピーして共有してもらえれば、リポジトリへ追記できます。


In [ ]:
from datetime import datetime, timezone, timedelta

jst = timezone(timedelta(hours=9))
now = datetime.now(jst).strftime("%Y-%m-%d %H:%M JST")

lines = [
    "# 実行記録: 01 L4確認 + 小さいモデル",
    "",
    f"- 実行日: {now}",
    "- 実行場所: Google Colab",
    f"- GPU: {gpu_name}",
    f"- VRAM: {round(vram_gb, 1)} GB",
    f"- L4だったか: {'はい' if gpu_ok else 'いいえ'}",
    f"- モデル: {MODEL_ID}",
    f"- 質問: {prompt}",
    "",
    "## モデルの返事",
    "",
    "```",
    reply.strip(),
    "```",
    "",
    "## メモ",
    "",
    "- 目的は賢い返事ではなく、公開モデルが自分のColabで動いたこと",
    "- 次は、もう少し大きいモデルか、同じモデルへの追加学習を検討する",
]
report = "\n".join(lines)
print(report)


## 終わったら

GPUをつないだまま放置すると、回数券（CU）が減ります。

- 今日はここまでなら **ランタイム → セッションを管理 → 解放**
- うまくいったら、上の実行記録をチャットに貼る
